In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.schema import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

import os
import json


c:\Users\Rakesh Kumar\VSCode\Medical_Assistant\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# STEP 1: Load medicine JSON
with open("../datasets/rag_json_docs/allopathy_drugs.json", "r", encoding="utf-8") as f:
    medicines = json.load(f)


# STEP 2: Convert medicine JSON → semantic text
def medicine_to_text(m):
    return f"""
MEDICATION: {m.get('entity', '')}

CLASS: {m.get('drug_class', '')}

DESCRIPTION:
{m.get('description', '')}

USES:
{", ".join(m.get('uses', []))}

MECHANISM OF ACTION:
{m.get('mechanism', '')}

DOSAGE:
Adults: {m.get('dosage', {}).get('adults', '')}
Children: {m.get('dosage', {}).get('children', '')}

ONSET:
{m.get('onset', '')}

DURATION:
{m.get('duration', '')}

FORMS:
{", ".join(m.get('forms', []))}

SIDE EFFECTS:
{", ".join(m.get('side_effects', []))}

WARNINGS:
{", ".join(m.get('warnings', []))}

INTERACTIONS:
{", ".join(m.get('interactions', []))}

COMBINATION NOTE:
{m.get('combination_note', '')}
""".strip()


# STEP 3: Create LangChain Documents
documents = []

for med in medicines:
    if not isinstance(med, dict):
        continue  # skip bad rows

    doc = Document(
        page_content=medicine_to_text(med),
        metadata={
            "medication": med.get("entity"),
            "class": med.get("drug_class"),
            "uses": med.get("uses", []),
            "aliases": med.get("ocr_aliases", []),
            "tags": med.get("uses", []) + [med.get("drug_class", "")]
        }
    )

    documents.append(doc)

In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

vector_db = FAISS.from_documents(documents, embeddings)

retriever = vector_db.as_retriever()

vector_db.save_local("vector_db/allopathy_drugs_db")

C:\Users\Rakesh Kumar\AppData\Local\Temp\ipykernel_29612\3333191641.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2972.79it/s]
